# Lab W6: Demonstrasi Temporal Leakage

Kamu membangun dua pipeline time series - causal (split kronologis, rolling feature di-shift sebelum agregasi) dan leaky - lalu mengukur inflasi metrik akibat temporal leakage dan mendiagnosis sumbernya. Lab ini utama untuk W6. Konsep (§) merujuk ke `06_W6_Representations_Temporal_Leakage.md`.

**Prasyarat:** Bab W6 §2 (temporal leakage + contoh konkret), §2.2 (5 jenis leakage), §2.4 (preprocessing leakage) sudah dibaca, familiar dengan time series, RandomForestClassifier (sklearn), serta `df.rolling()` dan `df.shift()` di pandas. **Hardware & waktu:** CPU cukup, ~1-2 jam.

## Alur Lab

1. **Satu deret waktu:** buat fitur yang hanya melihat masa lalu.
2. **Split kronologis:** pisahkan train dan validasi sesuai waktu.
3. **Pipeline causal:** hitung metrik tanpa informasi masa depan.
4. **Pipeline leaky:** ulangi dengan fitur yang membocorkan masa depan.
5. **Leakage inflation:** ukur selisih metrik dan jelaskan risikonya.

## 1. Setup

Dataset sintetis berupa tren sinusoidal yang berubah lambat ditambah noise, dengan label biner yang ditentukan dari median nilai di masa depan.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

np.random.seed(42)

def generate_temporal_dataset(n_samples=2000, noise=0.3):
    """Generate time series with temporal dependency.
    Target: 1 if current value > rolling median of FUTURE 20 values.
    """
    t = np.linspace(0, 40 * np.pi, n_samples)
    trend = np.sin(t) * 0.5 + np.sin(t * 0.15) * 0.3
    value = trend + np.random.randn(n_samples) * noise

    # Label: future-looking (THIS IS THE LEAK)
    future_median = pd.Series(value).rolling(20, min_periods=1).median().shift(-10).values
    y = (value > future_median).astype(int)

    df = pd.DataFrame({"value": value, "trend": trend, "y": y})
    return df

df = generate_temporal_dataset(2000, noise=0.3)
print(f"Shape: {df.shape}")
print(f"Rasio kelas: {df.y.mean():.3f} / {1-df.y.mean():.3f}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
ax1.plot(df["trend"], label="Tren asli", alpha=0.7)
ax1.plot(df["value"], label="Nilai teramati", alpha=0.4)
ax1.set_title("Deret Waktu (300 titik pertama)")
ax1.set_xlabel("Waktu"); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_xlim(0, 300)

ax2.scatter(range(len(df)), df["y"], s=5, c=df["y"], cmap="coolwarm", alpha=0.5)
ax2.set_title("Label (merah=1, biru=0) - 300 titik pertama")
ax2.set_xlabel("Waktu"); ax2.set_ylabel("Kelas")
ax2.set_xlim(0, 300)
plt.tight_layout(); plt.show()

## 2. Pembuatan Rolling Feature

Kita buat rolling feature dengan dua cara. Pipeline causal melakukan shift sebelum rolling sehingga hanya memakai masa lalu, sedangkan pipeline leaky melakukan rolling langsung sehingga membocorkan masa depan.

In [ ]:
def make_features_causal(df, window=10):
    """Rolling features: hanya dari masa lalu (shift 1)."""
    feat = pd.DataFrame(index=df.index)
    feat["value_lag1"] = df["value"].shift(1)
    feat["value_lag3"] = df["value"].shift(3)
    feat["value_lag5"] = df["value"].shift(5)
    feat["roll_mean"] = df["value"].shift(1).rolling(window, min_periods=3).mean()
    feat["roll_std"] = df["value"].shift(1).rolling(window, min_periods=3).std()
    feat["roll_min"] = df["value"].shift(1).rolling(window, min_periods=3).min()
    feat["roll_max"] = df["value"].shift(1).rolling(window, min_periods=3).max()
    feat["diff1"] = df["value"].diff(1)
    feat["diff3"] = df["value"].diff(3)
    feat["trend"] = df["trend"]
    return feat

def make_features_leaky(df, window=10):
    """Rolling features: TANPA shift - bocor informasi masa depan!"""
    feat = pd.DataFrame(index=df.index)
    feat["value_lag1"] = df["value"].shift(1)
    feat["roll_mean"] = df["value"].rolling(window, min_periods=3).mean()
    feat["roll_std"] = df["value"].rolling(window, min_periods=3).std()
    feat["roll_min"] = df["value"].rolling(window, min_periods=3).min()
    feat["roll_max"] = df["value"].rolling(window, min_periods=3).max()
    feat["future_mean"] = df["value"].shift(-5).rolling(5, min_periods=2).mean()
    feat["diff1"] = df["value"].diff(1)
    feat["trend"] = df["trend"]
    return feat

feat_causal = make_features_causal(df)
feat_leaky = make_features_leaky(df)

print(f"Causal features: {feat_causal.shape[1]} cols, {feat_causal.isna().sum().sum()} NaN")
print(f"Leaky features:  {feat_leaky.shape[1]} cols, {feat_leaky.isna().sum().sum()} NaN")
print("\nCausal feature columns:")
for c in feat_causal.columns:
    print(f"  {c}")
print("\nLeaky feature columns (extra leakage):")
leaky_extra = set(feat_leaky.columns) - set(feat_causal.columns)
for c in feat_leaky.columns:
    if c in leaky_extra:
        print(f"  [LEAK] {c}")

## 3. Pipeline Causal

Kita split data secara kronologis: 80% pertama menjadi train, 20% terakhir menjadi test. Baris NaN dibuang, lalu RandomForest dilatih.

In [ ]:
def chronological_split(X, y, ratio=0.8):
    n = len(X)
    split = int(n * ratio)
    return X[:split], X[split:], y[:split], y[split:]

X_c = feat_causal.dropna().values
y_c = df.loc[feat_causal.dropna().index, "y"].values
Xc_tr, Xc_te, yc_tr, yc_te = chronological_split(X_c, y_c)

clf_c = RandomForestClassifier(n_estimators=100, random_state=42)
clf_c.fit(Xc_tr, yc_tr)
pred_c = clf_c.predict(Xc_te)

acc_c = accuracy_score(yc_te, pred_c)
f1_c = f1_score(yc_te, pred_c)
print(f"=== CAUSAL PIPELINE === (split: kronologis)")
print(f"Train size: {len(Xc_tr)}, Test size: {len(Xc_te)}")
print(f"Accuracy: {acc_c:.4f}")
print(f"F1 Score: {f1_c:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
cm_c = confusion_matrix(yc_te, pred_c)
ConfusionMatrixDisplay(cm_c).plot(ax=ax)
ax.set_title("Pipeline Causal - Confusion Matrix")
plt.tight_layout(); plt.show()

## 4. Pipeline Leaky

Kita split data secara acak (shuffle) lalu membuat rolling feature tanpa shift. Ini kesalahan klasik: rolling tanpa shift membuat fitur memuat nilai masa depan, dan random split menyebarkan nilai itu ke train dan test.

In [ ]:
X_l = feat_leaky.dropna().values
y_l = df.loc[feat_leaky.dropna().index, "y"].values

Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(
    X_l, y_l, test_size=0.2, random_state=42, shuffle=True
)

clf_l = RandomForestClassifier(n_estimators=100, random_state=42)
clf_l.fit(Xl_tr, yl_tr)
pred_l = clf_l.predict(Xl_te)

acc_l = accuracy_score(yl_te, pred_l)
f1_l = f1_score(yl_te, pred_l)
print(f"=== LEAKY PIPELINE === (split: acak)")
print(f"Train size: {len(Xl_tr)}, Test size: {len(Xl_te)}")
print(f"Accuracy: {acc_l:.4f}")
print(f"F1 Score: {f1_l:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
cm_l = confusion_matrix(yl_te, pred_l)
ConfusionMatrixDisplay(cm_l).plot(ax=ax)
ax.set_title("Pipeline Leaky - Confusion Matrix")
plt.tight_layout(); plt.show()

## 5. Leakage Inflation

Kita hitung leakage inflation = F1_leaky dikurangi F1_causal. Makin besar selisihnya, makin berbahaya kebocorannya.

In [ ]:
print("=== LEAKAGE INFLATION REPORT ===")
print(f"\nPipeline          Accuracy     F1")
print(f"---               --------     --")
print(f"Causal  (chrono)  {acc_c:.4f}      {f1_c:.4f}")
print(f"Leaky   (random)  {acc_l:.4f}      {f1_l:.4f}")
print(f"---")
print(f"Inflation        {acc_l-acc_c:+.4f}      {f1_l-f1_c:+.4f}")

infl_abs = f1_l - f1_c
infl_rel = (f1_l - f1_c) / max(f1_c, 0.001)  # rel ke F1 causal
print(f"\n=== LEAKAGE INFLATION = {infl_abs:.4f} (abs) = {infl_rel*100:.1f}% (rel) ===")

# Threshold per module W6: >=0.05 abs OR >=10% rel = significant; <0.02 abs = noise
if infl_abs >= 0.05 or infl_rel >= 0.10:
    print(f">>> SIGNIFIKAN: Inflasi abs={infl_abs:.4f} atau rel={infl_rel*100:.1f}% melebihi threshold.")
    print(">>> Temporal leakage mendominasi performa. Perbaiki pipeline sebelum eksperimen apapun.")
elif infl_abs < 0.02:
    print(">>> AMAN: Inflasi < 0.02 (noise). Pipeline cukup aman dari temporal leakage.")
else:
    print(f">>> WASPADA: Inflasi {infl_abs:.4f} di antara 0.02-0.05. Periksa fitur yang mungkin leak.")

feature_names = list(feat_leaky.dropna().columns)
importances = clf_l.feature_importances_
top_idx = np.argsort(importances)[-5:]
print("\nTop-5 fitur paling penting di leaky pipeline:")
for i in top_idx:
    leak_flag = "[LEAK]" if "roll_" in feature_names[i] or "future" in feature_names[i] else ""
    print(f"  {feature_names[i]:20s} {importances[i]:.4f}  {leak_flag}")

## 6. Diagnostik

Kita plot prediksi leaky dan causal pada test set. Model leaky biasanya unggul di titik perubahan tren karena fiturnya sudah memuat informasi masa depan.

In [ ]:
idx_causal = feat_causal.dropna().index[-len(pred_c):]
df_causal = df.loc[idx_causal].copy()
df_causal["pred"] = pred_c

idx_leaky = feat_leaky.dropna().index[-len(pred_l):]
df_leaky = df.loc[idx_leaky].copy()
df_leaky["pred"] = pred_l

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df_causal.index, df_causal["y"], label="Label asli", color="black", alpha=0.7)
axes[0].set_ylabel("Label asli")
axes[0].set_title("Label Asli")
axes[0].grid(alpha=0.3)

correct_c = df_causal["y"] == df_causal["pred"]
axes[1].plot(df_causal.index, df_causal["pred"], label="Prediksi causal", color="blue", alpha=0.7)
axes[1].fill_between(df_causal.index, 0, 1, where=~correct_c, color="red", alpha=0.2, label="Error")
axes[1].set_ylabel("Prediksi causal")
axes[1].set_title(f"Pipeline Causal (F1={f1_c:.3f})")
axes[1].grid(alpha=0.3); axes[1].legend(loc="upper right")

correct_l = df_leaky["y"] == df_leaky["pred"]
axes[2].plot(df_leaky.index, df_leaky["pred"], label="Prediksi leaky", color="red", alpha=0.7)
axes[2].fill_between(df_leaky.index, 0, 1, where=~correct_l, color="red", alpha=0.15, label="Error")
axes[2].set_ylabel("Prediksi leaky")
axes[2].set_xlabel("Waktu")
axes[2].set_title(f"Pipeline Leaky (F1={f1_l:.3f})")
axes[2].grid(alpha=0.3); axes[2].legend(loc="upper right")

plt.tight_layout(); plt.show()

print("Apa pola yang terlihat? Leaky pipeline biasanya lebih akurat di sekitar")
print("titik perubahan tren, karena rolling window-nya memuat nilai masa depan.")

## 7. Kenapa Ini Berbahaya?

Model leaky mendapat F1 tinggi di validasi, tetapi saat deploy performanya turun tajam karena tidak ada data masa depan. Saat training, rolling window memuat nilai dari masa depan; saat deploy, window hanya punya data masa lalu.

**Intinya:** accuracy atau F1 yang tinggi di hari pertama bukan kabar baik, justru alasan untuk curiga.

### Tugas

Tulis satu paragraf: apa yang membuat angka leaky terlihat meyakinkan, dan mengapa angka itu tetap salah?

In [ ]:
# Tulis jawaban Anda di sel markdown di atas.
print("Sel sengaja kosong.")

## 8. Refleksi

1. **Ukuran window.** Coba ulangi dengan `window=3` vs `window=50`. Apakah leakage inflation berubah? Kenapa?

2. **Deteksi dini.** Dari EDA berlapis di W6, lapis mana yang pertama menangkap temporal leakage?

3. **Koneksi ke Capstone.** Dataset Capstone Anda mungkin punya komponen temporal. Tuliskan cara memverifikasi tidak ada temporal leakage di pipeline Capstone Anda.

## Self-Check Quick

- [ ] Dua pipeline dibangun: causal (split kronologis + rolling dengan shift) dan leaky (split acak + rolling tanpa shift).
- [ ] Leakage inflation dihitung: F1_leaky dikurangi F1_causal.
- [ ] Ambang dicek: inflation >= 0.05 absolut atau >= 10% relatif ditandai signifikan.
- [ ] Satu paragraf "kenapa angka leaky menipu" ditulis.